# Matmul: PMPP/Ch. 5

Tiled matmul with shared memory in CUDA and Triton + benchmarking

In [ ]:
import cupy as cp
import numpy as np
import torch
import triton
import triton.language as tl
import time

print(f'GPU: {cp.cuda.Device().name}')

WIDTH = 256
M_cp = cp.eye(WIDTH, dtype=cp.float32)
N_cp = cp.eye(WIDTH, dtype=cp.float32)
M_t  = torch.eye(WIDTH, device='cuda', dtype=torch.float32)
N_t  = torch.eye(WIDTH, device='cuda', dtype=torch.float32)

![image.png](../imgs/ch5mm.png)

Each block now uses two scratchpads, Mds and Nds, to pull chunks of M and N into shared memory. All threads in the block then take elements from there instead of global memory for every multiply. Each value gets reused 16 (=Tile) times, which also reduces global memory reads by a factor of 16.

Each block pulls one tile at a time, pulling tiles row-wise from M and column-wise from N. 

The syncthreads() keyword is needed twice per tile: once after loading (so no thread starts computing before the tile is fully loaded) and once after computing (so no thread overwrites elements before all threads in a block finished).

#### Tiled matmul (CUDA)

In [ ]:
TILE = 16

matmul_tiled_cuda = cp.RawKernel(rf'''
extern "C" __global__
void matmul_tiled_cuda(const float *M, const float *N, float *P, int width) {{
    __shared__ float Mds[{TILE}][{TILE}];
    __shared__ float Nds[{TILE}][{TILE}];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * {TILE} + ty;
    int col = blockIdx.x * {TILE} + tx;

    float sum = 0.0f;

    for (int t = 0; t < width / {TILE}; t++) {{
        Mds[ty][tx] = M[row * width + (t * {TILE} + tx)];
        Nds[ty][tx] = N[(t * {TILE} + ty) * width + col];

        __syncthreads();

        for (int k = 0; k < {TILE}; k++)
            sum += Mds[ty][k] * Nds[k][tx];

        __syncthreads();
    }}

    if (row < width && col < width)
        P[row * width + col] = sum;
}}
''', 'matmul_tiled_cuda')

GRID = WIDTH // TILE
P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)

matmul_tiled_cuda(
    (GRID, GRID), (TILE, TILE),
    (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH))
)
assert cp.allclose(P, cp.eye(WIDTH)), 'WRONG'
print(f'Tiled CUDA: correct')
print(f'Global memory reads reduced by {TILE}x vs thread-per-elem')

#### Tiled matmul (Triton)

tl.dot handles the shared memory tile internally, so no explicit "shared" or "syncthreads" keywords are needed.

In [ ]:
@triton.jit
def matmul_tiled_triton(M_ptr, N_ptr, P_ptr, width, TILE: tl.constexpr):
    row = tl.program_id(1) * TILE + tl.arange(0, TILE)[:, None]
    col = tl.program_id(0) * TILE + tl.arange(0, TILE)[None, :]

    acc = tl.zeros((TILE, TILE), dtype=tl.float32)

    for t in range(0, width, TILE):
        k = t + tl.arange(0, TILE)

        m = tl.load(M_ptr + row * width + k[None, :],
                    mask=(row < width) & (k[None, :] < width), other=0.0)
        n = tl.load(N_ptr + k[:, None] * width + col,
                    mask=(k[:, None] < width) & (col < width), other=0.0)

        acc += tl.dot(m, n)

    tl.store(P_ptr + row * width + col, acc,
             mask=(row < width) & (col < width))


GRID_T = WIDTH // TILE
P_t = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)

matmul_tiled_triton[(GRID_T, GRID_T)](
    M_t, N_t, P_t, WIDTH, TILE=TILE
)
assert torch.allclose(P_t, torch.eye(WIDTH, device='cuda')), 'WRONG'
print(f'Tiled Triton: correct')

#### Benchmark

In [ ]:
RUNS = 10

def bench_cuda(fn, grid, block, args):
    fn(grid, block, args)
    cp.cuda.stream.get_current_stream().synchronize()
    t0 = time.time()
    for _ in range(RUNS):
        fn(grid, block, args)
    cp.cuda.stream.get_current_stream().synchronize()
    return (time.time() - t0) / RUNS * 1000

def bench_triton(fn, grid, *args, **kwargs):
    fn[grid](*args, **kwargs)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(RUNS):
        fn[grid](*args, **kwargs)
    torch.cuda.synchronize()
    return (time.time() - t0) / RUNS * 1000

# Ch. 4 thread-per-elem baseline for comparison
tpe_cuda = cp.RawKernel(r'''
extern "C" __global__
void tpe(const float *M, const float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < width && col < width) {
        float sum = 0.0f;
        for (int k = 0; k < width; k++)
            sum += M[row * width + k] * N[k * width + col];
        P[row * width + col] = sum;
    }
}
''', 'tpe')

P_out   = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
P_out_t = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)

t_tpe_cuda   = bench_cuda(tpe_cuda,          (GRID, GRID),   (TILE, TILE), (M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH)))
t_tiled_cuda = bench_cuda(matmul_tiled_cuda, (GRID, GRID),   (TILE, TILE), (M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH)))
t_tiled_triton = bench_triton(matmul_tiled_triton, (GRID_T, GRID_T), M_t, N_t, P_out_t, WIDTH, TILE=TILE)

print(f'Thread-per-elem CUDA (ch4 baseline): {t_tpe_cuda:.2f} ms')
print(f'Tiled CUDA:                          {t_tiled_cuda:.2f} ms')
print(f'Tiled Triton:                        {t_tiled_triton:.2f} ms')
print(f'\nSpeedup tiled vs thread-per-elem: {t_tpe_cuda / t_tiled_cuda:.1f}x')